# Olist: Does Late Delivery Drive Low Review Scores — and Why?

**The question.** Olist is a Brazilian e-commerce marketplace with ~100K orders and
customer reviews. Do late deliveries actually drive low review scores — and if so, what
exactly are customers upset about?

**The answer, up front.** Decisively yes. On-time orders average **4.30★**; late orders
average **2.57★** — a 1.73-star collapse, with late orders 46% 1-star. And an LLM reading
the negative reviews shows *why*: the complaints are overwhelmingly about the **parcel**
(never arrived, arrived late), not the product.

**How it's built — three layers, each cross-validating the next:**
1. **SQL · bronze → silver → gold (DuckDB).** Joins and grain resolution to *one row per
   review*; undelivered orders excluded, items/payments pre-aggregated to order grain.
2. **Sentiment · two methods.** A Portuguese transformer and LeIA (Portuguese VADER), each
   validated against the 1–5★ ground truth — external validity kept separate from
   model-vs-model agreement; disagreements inspected, not dropped.
3. **Generative theming · LLM.** Structured output (theme + severity per review) re-joins
   the dataframe and drives the theme chart; a stability check guards against hallucination.

**Reproducible.** Expensive model outputs are cached and keyed to `review_id`, so the
notebook runs top-to-bottom from cache with no API calls.

*Data: Olist Brazilian E-Commerce Public Dataset (Kaggle).*

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

sns.set_theme(style="whitegrid")
OUTCOME_ORDER   = ["On-time", "Late"]
OUTCOME_PALETTE = {"On-time": "#4c9f70", "Late": "#d1495b"}

# Dark-theme palette shared by every chart
BG, FG, FG2, GRID_CLR = "#1D1D20", "#fbfbff", "#909094", "#2e2e33"

REGENERATE_SENTIMENT = False  # rebuild needs torch
REGENERATE_THEMES    = False  # rebuild needs anthropic + API key
print("setup ready · duckdb", duckdb.__version__, "· pandas", pd.__version__)

### 1 · Bronze — scope the raw data
The dataset ships **nine** tables. This analysis follows a single spine — *delivery →
reviews* — so Bronze scopes to the six that serve it (`orders`, `order_reviews`,
`order_items`, `order_payments`, `customers`, `geolocation`) and names the three it
deliberately sets aside (`products`, `sellers`, `product_category_name_translation`).
Nothing is silently dropped.

In [ ]:
BRONZE = {
    "orders":         "olist_orders_dataset",
    "order_reviews":  "olist_order_reviews_dataset",
    "order_items":    "olist_order_items_dataset",
    "order_payments": "olist_order_payments_dataset",
    "customers":      "olist_customers_dataset",
    "geolocation":    "olist_geolocation_dataset",
}

counts = {k: duckdb.sql(f"SELECT COUNT(*) FROM read_csv_auto('{v}.csv')").fetchone()[0]
          for k, v in BRONZE.items()}
print("Bronze scoped to spine:")
for k, n in counts.items():
    print(f"  {k:15} {n:>9,}")

### 2 · Silver — clean, resolve grain, pre-aggregate
Every grain trap is settled here, so the review grain downstream is never corrupted:

- **Exclusions.** Orders with no delivery date are removed — lateness is undefined without
  one. The criterion is the null *date*, not "canceled" status: verified, because the 2,965
  null-date orders span seven statuses and don't equal the 625 canceled ones.
- **Review grain.** `review_id` isn't unique in the raw data (814 duplicate rows, and some
  IDs map to multiple orders), so reviews are deduped to one row per `review_id` among
  delivered orders, with `order_id` as a deterministic tiebreaker.
- **Order-grain aggregation.** Items and payments are collapsed to one row per order
  *before* they meet the review grain — otherwise a 5-item order would count its delivery
  lateness five times and every average would quietly break.

Each silver table is written to Parquet, so the medallion is inspectable on disk.

In [ ]:
con = duckdb.connect()
for view, csv in {
    "orders":         "olist_orders_dataset",
    "order_reviews":  "olist_order_reviews_dataset",
    "order_items":    "olist_order_items_dataset",
    "order_payments": "olist_order_payments_dataset",
    "geolocation":    "olist_geolocation_dataset",
}.items():
    con.execute(f"CREATE VIEW {view} AS SELECT * FROM read_csv_auto('{csv}.csv')")

con.execute("""
CREATE TABLE orders_clean AS
  SELECT order_id, customer_id, order_status, order_purchase_timestamp,
         order_delivered_customer_date, order_estimated_delivery_date
  FROM orders WHERE order_delivered_customer_date IS NOT NULL;

CREATE TABLE reviews_dedup AS
  SELECT * EXCLUDE (rn) FROM (
    SELECT rev.*, ROW_NUMBER() OVER (
             PARTITION BY rev.review_id
             ORDER BY (rev.review_comment_message IS NOT NULL) DESC,
                      rev.review_answer_timestamp DESC, rev.order_id) AS rn
    FROM order_reviews rev JOIN orders_clean USING (order_id)) WHERE rn = 1;

CREATE TABLE order_items_agg AS
  SELECT order_id, COUNT(*) n_items, COUNT(DISTINCT seller_id) n_sellers,
         SUM(price) items_total, SUM(freight_value) freight_total
  FROM order_items GROUP BY order_id;

CREATE TABLE order_payments_agg AS
  SELECT order_id, SUM(payment_value) payment_total, COUNT(*) n_payments,
         MAX(payment_installments) max_installments
  FROM order_payments GROUP BY order_id;

CREATE TABLE geolocation_centroid AS
  SELECT geolocation_zip_code_prefix zip_prefix, AVG(geolocation_lat) lat,
         AVG(geolocation_lng) lng, ANY_VALUE(geolocation_state) geo_state
  FROM geolocation GROUP BY geolocation_zip_code_prefix;
""")

for t in ["orders_clean", "reviews_dedup", "order_items_agg", "order_payments_agg", "geolocation_centroid"]:
    con.execute(f"COPY {t} TO '{t}.parquet' (FORMAT parquet)")
    print(f"{t:22} {con.sql(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>8,}  ->  {t}.parquet")
con.close()

### 3 · Gold — one row per review
The analytical table: **one row per review**, delivery facts joined on, order-grain
summaries attached. The inner join to `orders_clean` is what enforces the exclusion.
Result: 95,645 rows, unique on `review_id`.

> **🥇 The SQL / pandas seam.** SQL owns the data layer — set-based joins and grain
> resolution, structured bronze → silver → gold. Pandas owns the analytical layer — feature
> engineering, sentiment, theming, visualization. The split is by strength, and no table is
> ever built twice.

In [ ]:
gold = duckdb.sql("""
    SELECT r.review_id, r.order_id, r.review_score, r.review_comment_title, r.review_comment_message,
           (r.review_comment_message IS NOT NULL) AS has_text,
           o.order_purchase_timestamp, o.order_delivered_customer_date, o.order_estimated_delivery_date,
           c.customer_unique_id, c.customer_state, c.customer_zip_code_prefix, c.customer_city,
           g.lat AS customer_lat, g.lng AS customer_lng,
           i.n_items, i.n_sellers, i.items_total, i.freight_total,
           p.payment_total, p.max_installments
    FROM 'reviews_dedup.parquet' r
    JOIN 'orders_clean.parquet' o USING (order_id)
    JOIN read_csv_auto('olist_customers_dataset.csv') c USING (customer_id)
    LEFT JOIN 'order_items_agg.parquet' i USING (order_id)
    LEFT JOIN 'order_payments_agg.parquet' p USING (order_id)
    LEFT JOIN 'geolocation_centroid.parquet' g ON c.customer_zip_code_prefix = g.zip_prefix
""").df()
duckdb.sql("COPY (SELECT * FROM gold) TO 'gold.parquet' (FORMAT parquet)")
print(f"gold {len(gold):,} rows · unique review_id: {gold['review_id'].is_unique}  ->  gold.parquet")

### 4 · Features, and a hypothesis killed with evidence
Pandas derives the continuous **delivery gap** (estimated − actual, in days), the binary
**on-time/late** flag, and a **negative-review** flag (1–2★).

Before going further, one tempting angle is closed with evidence: **96.9% of customers order
exactly once.** There is no retention or lifetime-value story in this data — surfaced here
to *rule it out*, so the spine stays on delivery.

In [ ]:
df = duckdb.sql("SELECT * FROM 'gold.parquet'").df()
df["has_text"] = df["has_text"].astype(bool)
delivered = pd.to_datetime(df["order_delivered_customer_date"])
estimated = pd.to_datetime(df["order_estimated_delivery_date"])
df["delivery_gap_days"]  = (estimated - delivered).dt.total_seconds() / 86400
df["is_late"]            = df["delivery_gap_days"] < 0
df["delivery_outcome"]   = df["is_late"].map({False: "On-time", True: "Late"})
df["is_negative"]        = df["review_score"] <= 2
duckdb.sql("COPY (SELECT * FROM df) TO 'features.parquet' (FORMAT parquet)")
print(df.groupby("delivery_outcome")["review_score"].mean().round(2).to_dict())
print("late %.1f%% · negative %.1f%%" % (df["is_late"].mean()*100, df["is_negative"].mean()*100))

In [ ]:
pct = duckdb.sql("""
  WITH per AS (SELECT c.customer_unique_id, COUNT(*) n
               FROM read_csv_auto('olist_orders_dataset.csv') o
               JOIN read_csv_auto('olist_customers_dataset.csv') c USING (customer_id)
               GROUP BY 1)
  SELECT ROUND(100.0*COUNT(*) FILTER (WHERE n = 1)/COUNT(*), 1) FROM per
""").fetchone()[0]
print(f"{pct}% of customers order exactly once -> retention angle killed with evidence")

### 5 · The quantitative signal
Two charts establish late→bad on hard numbers, before any NLP. Chart 1 splits the review-
score distribution by delivery outcome; Chart 2 shows the delivery-gap distribution. Olist
pads its estimates (median ~12 days early), so the damage lives in the thin tail that
crosses zero into "late."

In [ ]:
df = duckdb.sql("SELECT * FROM 'features.parquet'").df()
prop = (df.groupby("delivery_outcome")["review_score"]
          .value_counts(normalize=True).rename("proportion").reset_index())

fig_review_score, ax = plt.subplots(figsize=(9, 5))
fig_review_score.patch.set_facecolor(BG)
ax.set_facecolor(BG)

sns.barplot(
    prop, x="review_score", y="proportion",
    hue="delivery_outcome",
    hue_order=OUTCOME_ORDER,
    palette=OUTCOME_PALETTE,
    ax=ax,
    width=0.6,
    edgecolor=BG,
    linewidth=1.0,
)

for container in ax.containers:
    labels = [f"{v:.0%}" if v > 0.01 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels,
                 label_type="edge", padding=4, fontsize=8.5, color=FG2)

ax.set_xlabel("Review Score (stars ★)", fontsize=11, labelpad=8, color=FG)
ax.set_ylabel("Share within outcome", fontsize=11, labelpad=8, color=FG)
ax.set_ylim(0, ax.get_ylim()[1] * 1.14)
ax.tick_params(axis="both", labelsize=10, colors=FG2)

_yticks = ax.get_yticks()
ax.set_yticks(_yticks)
ax.set_yticklabels([f"{v:.0%}" for v in _yticks], color=FG2)

ax.grid(axis="y", color=GRID_CLR, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_edgecolor(GRID_CLR)

ax.set_title("Late deliveries collapse into 1-star reviews",
             fontsize=14, fontweight="bold", pad=16, loc="left", color=FG)
ax.legend(title="Delivery outcome", title_fontsize=9, fontsize=9,
          frameon=False, loc="upper left", labelcolor=FG)

plt.tight_layout()
plt.savefig("fig_review_score.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
plt.close("all")

In [ ]:
gap  = df["delivery_gap_days"].clip(-30, 60)
bins = list(range(-30, 61, 1))

fig_histogram, ax = plt.subplots(figsize=(9, 5))
fig_histogram.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.hist(
    [gap[~df["is_late"]], gap[df["is_late"]]],
    bins=bins, stacked=True,
    color=[OUTCOME_PALETTE["On-time"], OUTCOME_PALETTE["Late"]],
    label=OUTCOME_ORDER,
    edgecolor=BG, linewidth=0.4,
)

ax.axvline(0, color=FG2, linestyle="--", linewidth=1.2, label="Estimated date")

ax.set_xlabel("Delivery gap (days)  —  positive = arrived early, negative = arrived late",
              fontsize=10.5, labelpad=8, color=FG)
ax.set_ylabel("Orders", fontsize=10.5, labelpad=8, color=FG)
ax.tick_params(axis="both", labelsize=9.5, colors=FG2)

_yticks = ax.get_yticks()
ax.set_yticks(_yticks)
ax.set_yticklabels([f"{int(v):,}" if v >= 0 else "" for v in _yticks], color=FG2)

ax.grid(axis="y", color=GRID_CLR, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_edgecolor(GRID_CLR)

ax.set_title("Most deliveries beat the estimate — the late tail is thin but toxic",
             fontsize=13, fontweight="bold", pad=14, loc="left", color=FG)
leg = ax.legend(title="Delivery", fontsize=9, title_fontsize=9, frameon=False, labelcolor=FG)
leg.get_title().set_color(FG2)

plt.tight_layout()
plt.savefig("fig_histogram.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
plt.close("all")

### 6 · Sentiment — two methods, validated against ground truth
Reviews are Portuguese free text. Two **independent** methods score them:

- **Method A — a Portuguese-native transformer** (`nlptown/bert-base-multilingual-uncased-sentiment`):
  reads Portuguese directly, emits a 1–5 star prediction.
- **Method B — LeIA**, a Portuguese adaptation of VADER: a lexicon — a deliberately
  *different architecture* from the transformer, so the two act as independent checks.
  (Stock English VADER returns near-zero garbage on Portuguese and is not used.)

**How it was built** (generated offline — the transformer needs `torch`; cached to
`sentiment.pkl` keyed to `review_id`):

```python
# Method A — Portuguese transformer → 1–5 stars
from transformers import pipeline
clf = pipeline("sentiment-analysis",
               model="nlptown/bert-base-multilingual-uncased-sentiment",
               truncation=True, max_length=512)
star_a = [int(r["label"].split()[0]) for r in clf(texts, batch_size=64)]

# Method B — LeIA (Portuguese VADER), lexicon-based
from LeIA import SentimentIntensityAnalyzer
leia = SentimentIntensityAnalyzer()
compound_b = [leia.polarity_scores(t)["compound"] for t in texts]

# Map both models AND the star ground truth to one neg/neu/pos scale, then compare
def stars_to_polarity(s):    return "negative" if s <= 2 else "neutral" if s == 3 else "positive"
def compound_to_polarity(c): return "positive" if c >= 0.05 else "negative" if c <= -0.05 else "neutral"

out = pd.DataFrame({"review_id": review_ids, "star_a": star_a,
                    "compound_b": compound_b, "review_score": scores})
out["label_a"]     = out["star_a"].map(stars_to_polarity)
out["label_b"]     = out["compound_b"].map(compound_to_polarity)
out["label_truth"] = out["review_score"].map(stars_to_polarity)
out["ab_agree"] = out["label_a"] == out["label_b"]      # internal agreement (A vs B)
out["a_valid"]  = out["label_a"] == out["label_truth"]  # external validity (A vs stars)
out["b_valid"]  = out["label_b"] == out["label_truth"]  # external validity (B vs stars)
```

In [ ]:
if REGENERATE_SENTIMENT:
    raise RuntimeError("Sentiment rebuild needs torch — keep REGENERATE_SENTIMENT=False")
sentiment = pd.read_pickle("sentiment.pkl")
print(f"sentiment: {sentiment.shape[0]:,} reviews scored (transformer + LeIA)")

> **🥇 Validated against stars, not against each other.** Two checks, kept explicitly
> separate. *External validity* is each model vs. the 1–5★ ground truth (A **75.0%**,
> B **64.3%**). *Internal agreement* is the models vs. each other (**64.4%**). They answer
> different questions — model-vs-model agreement is **not** "confidence." The ~13,800 rows
> where the models disagree are **inspected, not dropped**: that is where mixed-sentiment
> reviews live ("great product / terrible delivery"), the sharpest evidence for the spine.

In [ ]:
print("A external validity: %.1f%%" % (sentiment["a_valid"].mean()*100))
print("B external validity: %.1f%%" % (sentiment["b_valid"].mean()*100))
print("A-B internal agree : %.1f%%" % (sentiment["ab_agree"].mean()*100))
print("disagreement rows  :", int((~sentiment["ab_agree"]).sum()))
_txt = duckdb.sql("SELECT review_id, review_comment_message FROM 'features.parquet'").df()
_disagree = sentiment[~sentiment["ab_agree"]].merge(_txt, on="review_id")
print("\nmixed-sentiment examples (models split):")
for _, r in _disagree.sample(3, random_state=1).iterrows():
    print(f"  {r['review_score']}star | A={r['label_a']:>8} B={r['label_b']:>8} | "
          f"{str(r['review_comment_message'])[:68]}")

### 7 · Generative theming — what customers are actually angry about
Sentiment says *how* negative a review is; it doesn't say *why*. An LLM reads batches of
negative reviews and returns **structured output** — one theme + severity per review — from
a fixed vocabulary so labels stay comparable across batches. The structured result re-joins
the dataframe and drives the theme chart.

**How it was built** (runs against the Anthropic API — not available in this environment;
cached to `themes.pkl` keyed to `review_id`). This is the *one* provider-specific function,
with defensive JSON parsing:

```python
THEME_VOCAB = ["never_arrived", "late_delivery", "wrong_or_missing_item", "damaged_product",
               "not_as_described", "poor_quality", "refund_or_billing", "other"]

def get_themes(review_batch, model="claude-haiku-4-5"):
    """Theme one batch of negative reviews -> [{review_id, theme, severity}].
    The ONLY provider-specific code; everything downstream is provider-agnostic."""
    prompt = build_prompt(THEME_VOCAB, review_batch)     # asks for a JSON array, one row/review
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    for attempt in range(2):                             # one retry on malformed JSON
        text = client.messages.create(
            model=model, max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]).content[0].text
        try:
            rows = parse_json_array(text)                # strips markdown fences, grabs outer [...]
            for r in rows:                               # clamp unknown labels to "other"
                if r["theme"] not in THEME_VOCAB: r["theme"] = "other"
            return rows
        except Exception:
            if attempt == 1: raise

# Batching is forced by the context window; sampling is required for the stability check.
themes = pd.concat([pd.DataFrame(get_themes(b)) for b in batches(negatives, size=25)])
```

In [ ]:
if REGENERATE_THEMES:
    raise RuntimeError("Theming rebuild needs `pip install anthropic` + ANTHROPIC_API_KEY")
themes = pd.read_pickle("themes.pkl")
print(themes["theme"].value_counts().to_dict())

> **🥇 Structured output re-joins the dataframe.** The AI-assisted analytics workflow shown
> literally: the LLM's structured output re-joins Gold by `review_id` and drives the chart
> below — it never dead-ends as a prose blob. Verification is **stability**: run the same
> sample twice and a review lands on the same theme **95.7%** of the time — the answer to
> "how do you know it didn't hallucinate." The chart splits each theme on-time vs. late in a
> single frame: the delivery themes (`never_arrived`, `late_delivery`) carry almost all the
> "Late" mass, while product complaints sit on on-time orders.

In [ ]:
_features = duckdb.sql("SELECT review_id, is_late FROM 'features.parquet'").df()
_m = themes.merge(_features, on="review_id", how="left")
_m["delivery_outcome"] = _m["is_late"].map({False: "On-time", True: "Late"}).fillna("On-time")

_pivot = (
    _m.groupby(["theme", "delivery_outcome"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=OUTCOME_ORDER, fill_value=0)
)
_pivot = _pivot.loc[_pivot.sum(axis=1).sort_values().index]

_label_map = {
    "wrong_or_missing_item": "Wrong / Missing Item",
    "never_arrived":         "Never Arrived",
    "late_delivery":         "Late Delivery",
    "damaged_product":       "Damaged Product",
    "not_as_described":      "Not as Described",
    "poor_quality":          "Poor Quality",
    "refund_or_billing":     "Refund / Billing",
    "other":                 "Other",
}
_pivot.index = [_label_map.get(t, t.replace("_", " ").title()) for t in _pivot.index]

fig_themes = go.Figure()
for _outcome in OUTCOME_ORDER:
    if _outcome in _pivot.columns:
        fig_themes.add_trace(go.Bar(
            x=_pivot[_outcome], y=_pivot.index, name=_outcome,
            orientation="h",
            marker_color=OUTCOME_PALETTE[_outcome],
            marker_line_width=0,
        ))

fig_themes.update_layout(
    barmode="stack", bargap=0.25,
    title=dict(text="Why negative reviews are negative — delivery themes dominate",
               font=dict(size=14, color=FG), x=0),
    xaxis=dict(title="Reviews", title_font=dict(color=FG2, size=11),
               tickfont=dict(color=FG2, size=10), gridcolor="#2e2e33", zerolinecolor="#2e2e33"),
    yaxis=dict(title="", tickfont=dict(color=FG, size=10), gridcolor="#2e2e33"),
    legend=dict(title=dict(text="Delivery", font=dict(color=FG2, size=9)),
                font=dict(color=FG, size=9), bgcolor="rgba(0,0,0,0)"),
    plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(color=FG),
    margin=dict(l=160, r=20, t=60, b=50),
)
fig_themes.show()

### 8 · Where lateness concentrates
The map colors each city by its **volume-normalized late rate** (never raw counts, which
would just reproduce population) and sizes it by order volume. São Paulo — the seller hub —
is huge but pale; the red outliers sit far up the North/Northeast coast, where deliveries
travel farthest from the sellers. Cities under 30 orders are dropped so small-sample noise
doesn't masquerade as an outlier.

In [ ]:
df = duckdb.sql("SELECT * FROM 'features.parquet'").df()
agg = (df.groupby("customer_city")
         .agg(n=("review_id", "size"), late_rate=("is_late", "mean"),
              avg_score=("review_score", "mean"), lat=("customer_lat", "mean"),
              lng=("customer_lng", "mean"), state=("customer_state", "first"))
         .reset_index())
agg = agg[(agg["n"] >= 30) & agg["lat"].notna()]

agg["late_pct"]      = (agg["late_rate"] * 100).round(1).astype(str) + "%"
agg["avg_score_str"] = agg["avg_score"].round(2).astype(str)

fig_lateness_map = px.scatter_geo(
    agg, lat="lat", lon="lng", size="n", color="late_rate",
    color_continuous_scale="Reds",
    scope="south america",
    size_max=38,
    hover_name="customer_city",
    hover_data={
        "state": True, "late_pct": True, "avg_score_str": True,
        "n": True, "lat": False, "lng": False, "late_rate": False,
    },
    labels={"late_pct": "Late rate", "avg_score_str": "Avg score", "n": "Orders"},
    title="Where late deliveries concentrate (cities with ≥30 orders)",
)

fig_lateness_map.update_geos(
    fitbounds="locations", showcountries=True,
    bgcolor=BG, lakecolor=BG, landcolor="#2a2a2e",
    countrycolor="#3a3a40", showframe=False,
)

fig_lateness_map.update_layout(
    paper_bgcolor=BG,
    font=dict(color=FG, size=11),
    title=dict(font=dict(size=14, color=FG), x=0),
    coloraxis_colorbar=dict(
        title=dict(text="Late rate", font=dict(color=FG2, size=10)),
        tickfont=dict(color=FG2, size=9),
        tickformat=".0%",
        thickness=14, len=0.6,
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=700, autosize=True,
)
fig_lateness_map.show()

### Limitations & next steps
- **Predictive modeling left out deliberately** — the spine is explanatory, not predictive.
- **Seller segmentation stays a supporting cut** — it doesn't compound with the delivery story.
- **BI dashboarding out of scope** for a notebook deliverable (named because the JD lists it).
- **Next step:** statistical regional outlier detection — a **funnel plot** (late rate vs.
  volume with binomial control limits) or empirical-Bayes shrinkage — to flag states
  significantly worse than baseline given their volume.